# 1. Candidate–History Attention Reranker

Asks *which of the user's earlier clicks are relevant to this candidate?*

- **Article encoder**: frozen 64-dimensional text embeddings shared by candidates and history
- **Candidate–history attention**: multi-head attention with the candidate as the query and the
  earlier clicked articles as keys and values, so the history summary changes per candidate
- **Tabular branch** (optional): the frozen causal tabular features used by the MLP baseline
- **MLP scorer**: concatenates `[tabular ; candidate ; history context]` and outputs one logit

`USE_TABULAR` switches the tabular branch on or off; section 8 compares the two runs.
A grid over attention heads, dropout and scorer width is searched on validation and the
winner is tested.

## 1.1 Load Packages

In [ ]:
import json
import random
import os
import sys
from itertools import product
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import polars as pl
import pyarrow.parquet as pq
import torch
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

## 1.2 Set Up Paths

Set `ENVIRONMENT` to `"colab"` to read the project and data from Google Drive. The
repository supplies the shared evaluator; the data folder holds the prepared features
and receives every checkpoint, score file and metric table this notebook writes.

In [ ]:
ENVIRONMENT = "local"  # "local" or "colab"

if ENVIRONMENT == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/BT4222-Group-6-")
    DATA_ROOT = Path("/content/drive/MyDrive/v3-mindsmall")
else:
    PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path.cwd())).resolve()
    while not (PROJECT_ROOT / "5_Models").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent
    DATA_ROOT = Path(os.environ.get("DATA_ROOT", PROJECT_ROOT)).resolve()

# Ranking features, click labels and causal user histories from feature engineering
RUN = DATA_ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"

# Frozen article embeddings from the representation stage
REPRESENTATIONS = DATA_ROOT / "3_Representations" / "artifacts"

# Checkpoints, score files and metric tables written by the reranking notebooks
ARTIFACTS = DATA_ROOT / "5_Models" / "5_02_Reranking" / "artifacts"
RESULTS = DATA_ROOT / "results"

# The shared evaluator is the only project module the reranking notebooks import
sys.path.insert(0, str(PROJECT_ROOT / "5_Models" / "5_02_Reranking"))
from rr_evaluation import compare_results, rank_logged_slate, score_and_evaluate

## 1.3 Load Ranking Features

Each split is stored as Parquet partitions with one row per (impression, candidate)
pair. These previews read only the first rows; later cells load what they need.

In [ ]:
def preview_split(split, n_rows=5):
    """Read the first rows of a split without loading every partition."""
    first_part = sorted((RUN / f"ranking_features_{split}").glob("*.parquet"))[0]
    return next(pq.ParquetFile(first_part).iter_batches(batch_size=n_rows)).to_pandas()


train_preview = preview_split("train")
validation_preview = preview_split("validation")
train_preview.head()

In [ ]:
validation_preview.head()

# 2. Experiment Configuration

In [ ]:
SEED = 4222
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"Using {DEVICE}")

TRAIN_ROWS = 300_000
HISTORY_CAP = 50
EPOCHS = 5
BATCH_SIZE = 2048
WEIGHT_DECAY = 1e-5
SELECTION_METRIC = "ndcg_at_10"

GRID = {
    "num_heads": (1, 4),
    "dropout": (0.10, 0.20, 0.30),
    "hidden_dims": ((128, 64), (256, 128, 64)),
}
CONFIGS = [
    {"num_heads": heads, "dropout": dropout, "hidden_dims": hidden_dims}
    for heads, dropout, hidden_dims in product(GRID["num_heads"], GRID["dropout"], GRID["hidden_dims"])
]
RESUME = True   # reuse trials whose checkpoint and validation metrics already exist

In [ ]:
# The frozen feature schema lists the causal tabular inputs shared by every reranker
schema = json.loads((RUN / "feature_schema.json").read_text())
FEATURES = schema["model_columns"]
CATEGORICAL = [name for name in schema.get("categorical_columns", []) if name in FEATURES]
NUMERIC = [name for name in FEATURES if name not in CATEGORICAL]
print(f"{len(FEATURES)} tabular features: {len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical")

In [ ]:
USE_TABULAR = True      # False = text + click history only (the branch ablation)
BRANCHES = "tabular_text" if USE_TABULAR else "text"
TABULAR_COLUMNS = FEATURES if USE_TABULAR else []
print(f"Branches: {BRANCHES}")

# 3. Prepare Training Data

## 3.1 Sample Labelled Training Candidates

`sample_train_rows` draws a reproducible, uniform sample of training candidates. The
number of rows taken from each partition follows a hypergeometric draw, so the whole
training split never has to be loaded at once.

In [ ]:
def read_labelled_part(part, columns):
    """Join one training feature partition to its click labels."""
    features = pq.read_table(part, columns=["impression_uid", "news_code", *columns]).to_pandas()
    labels = pq.read_table(
        RUN / "labels_train" / part.name,
        columns=["impression_uid", "news_code", "clicked"],
    ).to_pandas()
    return features.merge(labels, on=["impression_uid", "news_code"], validate="one_to_one")


def sample_train_rows(limit, columns):
    """Sample `limit` labelled training candidates uniformly across all partitions."""
    parts = sorted((RUN / "ranking_features_train").glob("*.parquet"))
    sizes = [pq.ParquetFile(part).metadata.num_rows for part in parts]
    rng = np.random.default_rng(SEED)
    remaining_population = sum(sizes)
    remaining_take = min(limit, remaining_population)

    samples = []
    for part, size in zip(parts, sizes):
        take = int(rng.hypergeometric(size, remaining_population - size, remaining_take))
        if take:
            frame = read_labelled_part(part, columns)
            samples.append(frame.iloc[rng.choice(size, take, replace=False)])
        remaining_population -= size
        remaining_take -= take
    return pd.concat(samples, ignore_index=True)

## 3.2 Article Vectors and Causal Click Histories

Every article is represented by its frozen 64-dimensional text embedding. A user's
history is the list of articles they clicked **strictly before** the impression, saved
by feature engineering; at most the newest `HISTORY_CAP` clicks are used.

In [ ]:
# Map each news_code to its row in the frozen 64-dimensional text embedding matrix.
# Row 0 is reserved for padding.
article_index = pl.read_parquet(REPRESENTATIONS / "article_index.parquet", columns=["news_code", "article_row"])
news_to_row = dict(zip(article_index["news_code"].to_list(), article_index["article_row"].to_list()))
text_vectors = np.load(REPRESENTATIONS / "text64.npy", mmap_mode="r")
EMBEDDING_DIM = text_vectors.shape[1]
print(f"Article embeddings: {text_vectors.shape[0]:,} rows x {EMBEDDING_DIM} dimensions")

In [ ]:
def load_profiles(split, impression_ids=None):
    """Map each impression to the article rows its user clicked strictly before the impression."""
    profiles = (
        pl.scan_parquet(RUN / f"request_profiles_{split}" / "*.parquet")
        .select("impression_uid", "full_history_rows")
    )
    if impression_ids is not None:
        profiles = profiles.filter(pl.col("impression_uid").is_in(list(impression_ids)))
    profiles = profiles.collect()
    return dict(zip(profiles["impression_uid"].to_list(), profiles["full_history_rows"].to_list()))

In [ ]:
def encode_pairs(frame, profiles):
    """Return the padded click history and the candidate article row for every candidate.

    Histories keep the newest HISTORY_CAP clicks and are left-padded with row 0.
    """
    histories = np.zeros((len(frame), HISTORY_CAP), dtype=np.int64)
    candidates = np.empty(len(frame), dtype=np.int64)
    for index, (uid, news_code) in enumerate(zip(frame["impression_uid"], frame["news_code"])):
        history = profiles[uid][-HISTORY_CAP:]
        if len(history):
            histories[index, -len(history):] = history
        candidates[index] = news_to_row[news_code]
    return histories, candidates

## 3.3 Sample Training Candidates

The same sample is shared by every trial.

In [ ]:
train = sample_train_rows(TRAIN_ROWS, columns=TABULAR_COLUMNS)
train_profiles = load_profiles("train", train["impression_uid"].unique())
train_history, train_candidate = encode_pairs(train, train_profiles)
train_labels = train["clicked"].to_numpy(np.float32)
print(f"Training rows: {len(train):,}; click rate: {train_labels.mean():.3%}")

## 3.4 Tabular Branch

With `USE_TABULAR = False` the model receives a zero-width tabular input, so the
text-only run uses exactly the same architecture minus the tabular branch.

In [ ]:
def build_preprocessor():
    """Median-impute and standardise numeric inputs; mode-impute and one-hot encode categoricals."""
    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    categorical = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("numeric", numeric, NUMERIC),
        ("categorical", categorical, CATEGORICAL),
    ])

In [ ]:
preprocessor = build_preprocessor().fit(train[FEATURES]) if USE_TABULAR else None


def tabular_inputs(frame):
    """Preprocessed tabular features, or a zero-width array when the tabular branch is off."""
    if not USE_TABULAR:
        return np.zeros((len(frame), 0), dtype=np.float32)
    return preprocessor.transform(frame[FEATURES]).astype(np.float32, copy=False)

In [ ]:
train_tabular = tabular_inputs(train)
print(f"Tabular width: {train_tabular.shape[1]}")

# 4. Model: `CandidateAttentionReranker`

```text
Candidate text ─→ query ─┐
History text ──→ keys / values → multi-head attention → history context
                                         ↓
[tabular features ; candidate vector ; history context] → MLP → logit
```

Padding positions are masked. `nn.MultiheadAttention` cannot attend over a fully masked
sequence, so users with no history temporarily expose one padding slot and then receive
a zero context vector.

In [ ]:
class CandidateAttentionReranker(nn.Module):
    """Candidate-conditioned attention over causal text history, scored with optional tabular features."""

    def __init__(self, vectors, tabular_width, num_heads, hidden_dims, dropout):
        super().__init__()
        self.articles = nn.Embedding.from_pretrained(
            torch.as_tensor(np.asarray(vectors), dtype=torch.float32),
            freeze=True,
            padding_idx=0,
        )
        self.attention = nn.MultiheadAttention(
            embed_dim=EMBEDDING_DIM,
            num_heads=num_heads,
            batch_first=True,
        )

        layers, width = [], tabular_width + 2 * EMBEDDING_DIM
        for hidden in hidden_dims:
            layers.extend([nn.Linear(width, hidden), nn.ReLU(), nn.Dropout(dropout)])
            width = hidden
        layers.append(nn.Linear(width, 1))
        self.scorer = nn.Sequential(*layers)

    def forward(self, tabular, history_rows, candidate_rows):
        history = self.articles(history_rows)          # [batch, history, 64]
        candidate = self.articles(candidate_rows)      # [batch, 64]

        has_history = history_rows.ne(0).any(dim=1)
        padding_mask = history_rows.eq(0).clone()
        padding_mask[~has_history, 0] = False          # keep one slot open for empty histories

        context, _ = self.attention(
            query=candidate.unsqueeze(1),
            key=history,
            value=history,
            key_padding_mask=padding_mask,
            need_weights=False,
        )
        context = context.squeeze(1) * has_history.unsqueeze(1)   # zero context for cold users
        return self.scorer(torch.cat([tabular, candidate, context], dim=1)).squeeze(-1)

# 5. Training

Weighted binary cross-entropy, with the positive-class weight taken from the training click rate.

In [ ]:
def train_model(config):
    """Train one CandidateAttentionReranker configuration with weighted BCE."""
    torch.manual_seed(SEED)
    model = CandidateAttentionReranker(
        text_vectors, train_tabular.shape[1], config["num_heads"], config["hidden_dims"], config["dropout"],
    ).to(DEVICE)
    click_rate = train_labels.mean()
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([(1 - click_rate) / click_rate], device=DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=WEIGHT_DECAY)
    loader = DataLoader(
        TensorDataset(
            torch.from_numpy(train_tabular),
            torch.from_numpy(train_history),
            torch.from_numpy(train_candidate),
            torch.from_numpy(train_labels),
        ),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0
        for tabular, histories, candidates, labels in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(tabular.to(DEVICE), histories.to(DEVICE), candidates.to(DEVICE)), labels.to(DEVICE))
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(labels)
        print(f"  epoch {epoch + 1}/{EPOCHS}: train BCE={total_loss / len(train_labels):.5f}")
    return model

In [ ]:
def predict_batches(model, *inputs):
    """Score aligned NumPy inputs in mini-batches and return click probabilities."""
    model.eval()
    scores = []
    with torch.no_grad():
        for start in range(0, len(inputs[0]), BATCH_SIZE):
            batch = [torch.from_numpy(array[start:start + BATCH_SIZE]).to(DEVICE) for array in inputs]
            scores.append(torch.sigmoid(model(*batch)).cpu().numpy())
    return np.concatenate(scores)

In [ ]:
def attention_predictor(model, profiles):
    """Wrap a trained model and a split's histories as a predict(frame) function."""
    def predict(frame):
        histories, candidates = encode_pairs(frame, profiles)
        return predict_batches(model, tabular_inputs(frame), histories, candidates)
    return predict


def config_name(config):
    widths = "x".join(map(str, config["hidden_dims"]))
    return f"cand_attn_{BRANCHES}_h{config['num_heads']}_do{config['dropout']:.2f}_h{widths}"

# 6. Validation Grid Search

Each configuration is trained, saved and evaluated on the complete validation slates.
Rerunning the cell skips completed trials when `RESUME` is on.

In [ ]:
validation_profiles = load_profiles("validation")
grid_results = []
for index, config in enumerate(CONFIGS, start=1):
    name = config_name(config)
    output_dir = ARTIFACTS / name
    metrics_path = RESULTS / f"reranking_{name}_validation.csv"

    if RESUME and metrics_path.exists() and (output_dir / "model.pt").exists():
        print(f"[{index}/{len(CONFIGS)}] already complete: {name}")
        result = pl.read_csv(metrics_path)
    else:
        print(f"[{index}/{len(CONFIGS)}] running: {name}")
        model = train_model(config)
        output_dir.mkdir(parents=True, exist_ok=True)
        torch.save({"state_dict": model.state_dict(), "config": config, "use_tabular": USE_TABULAR,
                    "history_cap": HISTORY_CAP, "seed": SEED}, output_dir / "model.pt")
        if USE_TABULAR:
            joblib.dump(preprocessor, output_dir / "preprocessor.joblib")
        result, _ = score_and_evaluate(
            attention_predictor(model, validation_profiles),
            split="validation",
            feature_run=RUN,
            output_dir=output_dir,
            model_name=name,
            columns=TABULAR_COLUMNS,
            results_dir=RESULTS,
        )
    grid_results.append(result.with_columns(
        pl.lit(config["num_heads"]).alias("num_heads"),
        pl.lit(config["dropout"]).alias("dropout"),
        pl.lit("x".join(map(str, config["hidden_dims"]))).alias("hidden_dims"),
    ))

leaderboard = compare_results(grid_results, sort_by=SELECTION_METRIC)
leaderboard.write_csv(RESULTS / f"cand_attn_{BRANCHES}_validation_leaderboard.csv")
final_model_name = leaderboard["model"][0]
print(f"Selected using validation {SELECTION_METRIC}: {final_model_name}")
display(leaderboard.select("model", "num_heads", "dropout", "hidden_dims", "mrr", "ndcg_at_10").head(10))

# 7. Test Evaluation

Reload the validation winner's checkpoint and score every test candidate.

In [ ]:
final_dir = ARTIFACTS / final_model_name
checkpoint = torch.load(final_dir / "model.pt", map_location=DEVICE, weights_only=False)
final_config = checkpoint["config"]
final_model = CandidateAttentionReranker(
    text_vectors, train_tabular.shape[1], final_config["num_heads"], final_config["hidden_dims"], final_config["dropout"],
).to(DEVICE)
final_model.load_state_dict(checkpoint["state_dict"])

test_result, _ = score_and_evaluate(
    attention_predictor(final_model, load_profiles("test")),
    split="test",
    feature_run=RUN,
    output_dir=final_dir,
    model_name=final_model_name,
    columns=TABULAR_COLUMNS,
    results_dir=RESULTS,
)
display(test_result.select("model", "split", "mrr", "ndcg_at_5", "ndcg_at_10", "recall_at_10"))

# 8. Branch Ablation: With and Without Tabular Features

Run the notebook once with `USE_TABULAR = True` and once with `False`. This cell then
compares the best validation trial of each run: the gap is what the tabular features add on top of text and click history.

In [ ]:
branch_results = []
for branches in ("text", "tabular_text"):
    path = RESULTS / f"cand_attn_{branches}_validation_leaderboard.csv"
    if path.exists():
        branch_results.append(pl.read_csv(path).sort(SELECTION_METRIC, descending=True).head(1)
                              .with_columns(pl.lit(branches).alias("branches")))
if branch_results:
    display(compare_results(branch_results, sort_by=SELECTION_METRIC)
            .select("branches", "model", "mrr", "ndcg_at_5", "ndcg_at_10", "recall_at_10"))
else:
    print("Run the notebook with USE_TABULAR = True and False to fill this comparison.")

# 9. Recommendations for a User

`rank_logged_slate` reorders the articles shown in a user's latest validation
impression using this notebook's saved scores. It does not retrieve new articles
from the catalogue and never reads click labels.

In [ ]:
example_user = int(validation_preview["user_code"].iloc[0])
rank_logged_slate(example_user, score_file=final_dir / "validation_scores.parquet", feature_run=RUN, top_n=5)